# IRRM-CODEC: run and analyze

This notebook shows how to train from two inputs: an AIRR table and a parquet file with TCRemP embeddings.

## 1. Define inputs

The AIRR file must contain `clone_id`, `junction_aa`, `v_call`, `j_call`, `locus`.
The embeddings parquet must contain `clone_id` plus either `tcremp_emb` or numeric embedding columns.

In [ ]:
from pathlib import Path

root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
airr_path = root / 'data' / 'sample_airr.tsv'
embeddings_path = root / 'data' / 'sample_embeddings.parquet'
output_dir = root / 'artifacts' / 'forward_demo'

airr_path, embeddings_path, output_dir

## 2. Preview the two files

In [ ]:
import pandas as pd

airr_df = pd.read_csv(airr_path, sep='\t')
emb_df = pd.read_parquet(embeddings_path)

display(airr_df.head())
display(emb_df.head())

## 3. Launch training

In [ ]:
import subprocess

cmd = [
    'python', '-m', 'irrm_codec.train_forward',
    '--airr-path', str(airr_path),
    '--embeddings-path', str(embeddings_path),
    '--output-dir', str(output_dir),
    '--locus', 'alpha',
    '--epochs', '5',
]
# subprocess.run(cmd, cwd=root, check=True)
cmd

## 4. Inspect saved metrics and merge stats

In [ ]:
import json
import pandas as pd

history = json.loads((output_dir / 'history.json').read_text())
test_metrics = json.loads((output_dir / 'test_metrics.json').read_text())
data_stats = json.loads((output_dir / 'data_stats.json').read_text())
history_df = pd.json_normalize(history)

display(history_df)
display(test_metrics)
display(data_stats)

In [ ]:
history_df.plot(x='epoch', y=['train.loss', 'val.loss'], figsize=(8, 4), grid=True, title='Loss by epoch')

## 5. Load a checkpoint

In [ ]:
import torch
from irrm_codec.forward_model import ForwardModel

checkpoint = torch.load(output_dir / 'best.pt', map_location='cpu')
model = ForwardModel(output_dim=checkpoint['extra']['embedding_dim'])
model.load_state_dict(checkpoint['model_state'])
checkpoint['metrics']